In [1]:
import torch
import torch.nn as nn
import math

# Multi Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()

        self.heads = heads
        self.head_dim = d_model // heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        Q = Q.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.fc(out)


# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, heads)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        attn = self.attention(x)
        x = self.norm1(x + attn)

        ff = self.feed_forward(x)
        x = self.norm2(x + ff)

        return x


# Simple Transformer
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, heads):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.block = TransformerBlock(d_model, heads)

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)

        x = self.block(x)

        return self.fc(x)


# Example
x = torch.randint(0, 100, (2, 5))

model = Transformer(vocab_size=100, d_model=32, heads=4)

output = model(x)

print(output.shape)

torch.Size([2, 5, 100])
